# 第5课：分类任务实战

**学习目标：**
- 用神经网络完成一个完整的二分类任务
- 理解分类函数：将概率转换为类别标签
- 可视化分类结果
- 发现未训练网络的问题

---

现在我们有了完整的前向传播网络，是时候用它解决一个真实问题了：判断二维平面上的点属于哪个类别。

## 5.1 生成分类数据

我们用环形边界来生成二分类数据：圆环内的点为类别1，其余为类别0。

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '..')  # 添加上级目录到路径
from numpy.utils import create_data, plot_data

# 生成500个数据点
data = create_data(500)
print("数据形状:", data.shape)  # (500, 3) — x, y, label
print("前5个样本:")
print(data[:5])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plot_data(data, "真实分类（橙=0, 蓝=1）")

## 5.2 构建网络并前向传播

创建一个多层网络，输入2维坐标，输出2维概率（类别0和类别1的概率）。

In [ ]:
def activation_ReLU(x):
    return np.maximum(0, x)

def activation_softmax(inputs):
    max_val = np.max(inputs, axis=1, keepdims=True)
    shifted = inputs - max_val
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=1, keepdims=True)

class Layer:
    def __init__(self, n_inputs, n_neurons):
        self.weights = np.random.randn(n_inputs, n_neurons)
        self.biases = np.random.randn(n_neurons)

    def forward(self, inputs):
        self.sum = np.dot(inputs, self.weights) + self.biases
        return self.sum

class Network:
    def __init__(self, network_shape):
        self.shape = network_shape
        self.layers = []
        for i in range(len(network_shape) - 1):
            self.layers.append(Layer(network_shape[i], network_shape[i + 1]))

    def network_forward(self, inputs):
        outputs = [inputs]
        for i in range(len(self.layers)):
            z = self.layers[i].forward(outputs[i])
            if i == len(self.layers) - 1:
                a = activation_softmax(z)
            else:
                a = activation_ReLU(z)
            outputs.append(a)
        return outputs

## 5.3 分类函数

Softmax 输出的是概率，我们需要把它转换成类别标签（0 或 1）。方法很简单：取概率较大的那个类别。

In [ ]:
def classify(probabilities):
    """将概率转换为类别标签
    
    取第二列（类别1的概率），四舍五入得到 0 或 1
    """
    return np.rint(probabilities[:, 1]).astype(int)

# 示例
probs = np.array([[0.8, 0.2],   # 类别0概率高 → 预测0
                   [0.3, 0.7],   # 类别1概率高 → 预测1
                   [0.5, 0.5]])  # 相等 → 四舍五入
print("分类结果:", classify(probs))  # [0, 1, 0]

## 5.4 可视化未训练网络的输出

用随机权重初始化的网络做预测，看看效果如何：

In [ ]:
# 创建网络
net = Network([2, 3, 4, 2])

# 提取特征
inputs = data[:, :2]

# 前向传播
outputs = net.network_forward(inputs)
predictions = classify(outputs[-1])

# 构造预测结果数据
predicted_data = data.copy()
predicted_data[:, 2] = predictions

# 对比可视化
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plot_data(data, "真实分类")

plt.subplot(1, 2, 2)
plot_data(predicted_data, "未训练网络的预测")

plt.tight_layout()
plt.show()

可以看到，未训练的网络预测结果与真实分布完全不同 — 这是因为权重是随机的。

---

## 小结

- 分类函数：取 Softmax 输出中概率较大的类别
- 未训练的网络输出是随机的，没有意义
- **核心问题：如何调整权重让预测变准确？**

**下一课**我们将引入损失函数来量化"预测有多差"，为训练提供目标。